# Training and testing target-specific machine-learning models for structure-based virtual screening of PfDHODH
## 1. Import packages

In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.utils import parallel_backend
from xgboost.sklearn import XGBRegressor
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import math
import hebo
import optuna
import optunahub
from tqdm import tqdm
import timeit
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import rdMolDescriptors
import os

2026-03-31 17:00:20.997539: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Define a function to get train and test splits based on cluster IDs
def get_train_test_split(df, heldout_cluster):
    """
    heldout_cluster: cluster ID to use as test actives (0, 1, 2, 3)
    """
    # --- ACTIVE SPLIT ---
    train_actives = df[(df["activity"] == "Active") &
                       (df["Cluster_ID"] != heldout_cluster)]

    test_actives = df[(df["activity"] == "Active") &
                      (df["Cluster_ID"] == heldout_cluster)]

    # --- INACTIVE SPLIT ---
    train_inactives = df[(df["activity"] == "Inactive") & 
                         (df["Cluster_ID"] == -1)]

    all_decoys = df[(df["activity"] == "Inactive") &
                        (df["Cluster_ID"] != -1)]
    
    test_act_decoys = df[(df["activity"]=="Inactive") & 
                         (df["Cluster_ID"] == heldout_cluster)]

    # --- FINAL MERGING ---
    train_df = pd.concat([train_actives, train_inactives], ignore_index=True)
    full_test_df  = pd.concat([test_actives, all_decoys], ignore_index=True)
    test_act_decoys_test_df = pd.concat([test_actives, test_act_decoys], ignore_index=True)

    return train_df, full_test_df, test_act_decoys_test_df

# Define a function to load training and test features into separate np arrays
def split_features(df, train_df, test_df):
    " Provide a dataframe to split into training and test features based on train and test data molecule ids "
    train_features = df.loc[train_df['mol_name']].values
    test_features = df.loc[test_df['mol_name']].values
    return {"train_features": train_features, "test_features": test_features}

In [3]:
# Change the heldout cluster ID here (0, 1, 2, or 3)
heldout_cluster = 3

In [4]:
PfDHODH_data = pd.read_csv("../../data/PfDHODH-data.csv")
train_df, full_test_df, test_act_decoys_test_df = get_train_test_split(PfDHODH_data, heldout_cluster=heldout_cluster)

In [5]:
PLEC_df = pd.read_csv("../../data/features/PLEC_features.csv", dtype={'Molecule_Name': 'object'}, index_col = "Molecule_Name")
train_PLEC = split_features(PLEC_df, train_df, full_test_df)["train_features"]
test_PLEC = split_features(PLEC_df, train_df, full_test_df)["test_features"]
test_actdecoys_test_PLEC = split_features(PLEC_df, train_df, test_act_decoys_test_df)["test_features"]

train_Potency = train_df["potency"].values
test_Potency = full_test_df["potency"].values
test_actdecoys_test_Potency = test_act_decoys_test_df["potency"].values

train_Class = train_df["activity"].values
test_Class = full_test_df["activity"].values
test_actdecoys_test_Class = test_act_decoys_test_df["activity"].values

## Define a NEF10% scorer for the loss function

In [6]:
def nef_score(y_true, y_pred, top_percent=10):
    """
    NEF at top_percent% based on predicted potency.
    y_true: continuous potency values
    y_pred: predicted potency
    active_threshold: potency cutoff to define actives
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Define binary actives on the fly
    y_bin = (y_true >= 6).astype(int)
    
    n_mols = len(y_bin)
    n_actives = np.sum(y_bin)
    top_n = max(1, int(np.ceil(n_mols * top_percent / 100)))
    
    # Rank predictions descending
    idx_sorted = np.argsort(-y_pred)
    top_idx = idx_sorted[:top_n]
    
    n_actives_top = np.sum(y_bin[top_idx])
    ef_max = min(top_n, n_actives)
    
    nef = n_actives_top / ef_max if ef_max > 0 else 0
    return nef


## 6. Train and test ML SFs on PLEC features 
Loss function for hyperparameter tuning is the NEF10%
#### 6.1 Random Forest Regressor

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor

def objective(trial):
    # Hyperparameters
    n_estimators = trial.suggest_int("n_estimators", 100, 2000)
    criterion = trial.suggest_categorical(
        "criterion",
        ["squared_error", "friedman_mse", "poisson"]
    )

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        criterion=criterion,
        max_features="sqrt",
        n_jobs=40,
        random_state=42
    )

    # Binary labels ONLY for stratification
    y_bin = (train_Potency >= 6).astype(int)

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    nef_scores = []
    # mse_scores = []

    for train_idx, val_idx in cv.split(train_PLEC, y_bin):
        X_tr, X_val = train_PLEC[train_idx], train_PLEC[val_idx]
        y_tr, y_val = train_Potency[train_idx], train_Potency[val_idx]

        # Train regression model on potency
        model.fit(X_tr, y_tr)

        # Predict potency
        y_pred = model.predict(X_val)

        # Compute NEF on validation fold
        nef = nef_score(y_val, y_pred, top_percent=10)
        nef_scores.append(nef)

        # mse = mean_squared_error(y_val, y_pred)
        # mse_scores.append(mse)
    return float(np.mean(nef_scores))

module = optunahub.load_module("samplers/hebo")
sampler = module.HEBOSampler()

study = optuna.create_study(
    sampler=sampler,
    direction="maximize"
)

study.optimize(objective, n_trials=10)

print("Best params:", study.best_params)
print("Best CV NEF10%:", study.best_value)

[I 2026-03-31 17:00:29,078] A new study created in memory with name: no-name-74ee5394-7358-4ac1-a52a-389870895879
[I 2026-03-31 17:00:32,313] Trial 0 finished with value: 0.9545454545454546 and parameters: {'n_estimators': 227, 'criterion': 'squared_error'}. Best is trial 0 with value: 0.9545454545454546.
[I 2026-03-31 17:00:47,287] Trial 1 finished with value: 0.9696969696969697 and parameters: {'n_estimators': 1125, 'criterion': 'friedman_mse'}. Best is trial 1 with value: 0.9696969696969697.
[I 2026-03-31 17:01:03,377] Trial 2 finished with value: 0.9696969696969697 and parameters: {'n_estimators': 1231, 'criterion': 'friedman_mse'}. Best is trial 1 with value: 0.9696969696969697.
[I 2026-03-31 17:01:33,538] Trial 3 finished with value: 0.9696969696969697 and parameters: {'n_estimators': 1941, 'criterion': 'poisson'}. Best is trial 1 with value: 0.9696969696969697.
[I 2026-03-31 17:01:49,668] Trial 4 finished with value: 0.9696969696969697 and parameters: {'n_estimators': 982, 'crit

Best params: {'n_estimators': 1125, 'criterion': 'friedman_mse'}
Best CV NEF10%: 0.9696969696969697


In [8]:
# Evaluate on the full test set
for i in tqdm(range(10)):
    #Train the RF model on the training molecules, using optimal parameters:
    rf = RandomForestRegressor(n_estimators = study.best_params['n_estimators'], 
                               criterion = study.best_params['criterion'],
                               max_features = 'sqrt', n_jobs = 40, random_state=i)
    rf.fit(train_PLEC, train_Potency)
    #Test the RF model on the test molecules:
    predicted_potency = rf.predict(test_PLEC)
    
    #Get virtual screening results on the test molecules and export results to a csv file:
    result_rf = pd.DataFrame({"Predicted_Potency": predicted_potency,
                              "Real_Potency": test_Potency,
                              "Real_Class": test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_rf = result_rf.sort_values(by='Predicted_Potency',ascending=False)
    result_rf.to_csv("./screening-results/heldout_cluster_{}/PLEC/full_test_set/PfDHODH-rRF-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [01:02<00:00,  6.25s/it]


In [9]:
# Evaluate on the test set containing test active decoys only (no training actives decoys)
for i in tqdm(range(10)):
    #Train the RF model on the training molecules, using optimal parameters:
    rf = RandomForestRegressor(n_estimators = study.best_params['n_estimators'], 
                               criterion = study.best_params['criterion'],
                               max_features = 'sqrt', n_jobs = 40, random_state=i)
    rf.fit(train_PLEC, train_Potency)
    #Test the RF model on the test molecules:
    predicted_potency = rf.predict(test_actdecoys_test_PLEC)
    
    #Get virtual screening results on the test molecules and export results to a csv file:
    result_rf = pd.DataFrame({"Predicted_Potency": predicted_potency,
                              "Real_Potency": test_actdecoys_test_Potency,
                              "Real_Class": test_actdecoys_test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_rf = result_rf.sort_values(by='Predicted_Potency',ascending=False)
    result_rf.to_csv("./screening-results/heldout_cluster_{}/PLEC/smaller_test_set/PfDHODH-rRF-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [01:00<00:00,  6.09s/it]


#### 6.2 Extreme Gradient Boosting Regressor

In [10]:
def objective(trial: optuna.trial.Trial) -> float:
    # Define the search space using trial.suggest_* methods "
    n_estimators = trial.suggest_int("n_estimators", 100,1000)
    max_depth = trial.suggest_categorical("max_depth", [5, 6, 7, 8, 9, 10])
    
    # Use these parameters to train and evaluate your model
    model = XGBRegressor(n_estimators = n_estimators,
                         max_depth = max_depth,
                         n_jobs = 40)
    
    # Binary labels ONLY for stratification
    y_bin = (train_Potency >= 6).astype(int)

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    nef_scores = []
    # mse_scores = []

    for train_idx, val_idx in cv.split(train_PLEC, y_bin):
        X_tr, X_val = train_PLEC[train_idx], train_PLEC[val_idx]
        y_tr, y_val = train_Potency[train_idx], train_Potency[val_idx]

        # Train regression model on potency
        model.fit(X_tr, y_tr)

        # Predict potency
        y_pred = model.predict(X_val)

        # Compute NEF on validation fold
        nef = nef_score(y_val, y_pred, top_percent=10)
        nef_scores.append(nef)

        # mse = mean_squared_error(y_val, y_pred)
        # mse_scores.append(mse)

    return float(np.mean(nef_scores))

# Use HEBOSampler
module = optunahub.load_module("samplers/hebo")
sampler = module.HEBOSampler()

# Create and run the study
study = optuna.create_study(sampler=sampler, direction="maximize")
study.optimize(objective, n_trials=10)

print(study.best_trial.params, study.best_trial.value)

[I 2026-03-31 17:05:39,355] A new study created in memory with name: no-name-957c92d9-b3b8-4764-a670-235e5b282cca
[I 2026-03-31 17:06:48,694] Trial 0 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 715, 'max_depth': 9}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:08:10,102] Trial 1 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 996, 'max_depth': 10}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:09:08,742] Trial 2 finished with value: 0.8787878787878788 and parameters: {'n_estimators': 390, 'max_depth': 8}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:10:24,285] Trial 3 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 782, 'max_depth': 10}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:11:47,770] Trial 4 finished with value: 0.8636363636363636 and parameters: {'n_estimators': 897, 'max_depth': 6}. Best is trial 0 with value: 0.8939393

{'n_estimators': 715, 'max_depth': 9} 0.8939393939393939


In [11]:
#Train the XGB model on the training molecules, using optimal parameters:
for i in tqdm(range(10)):
    #Define the parameters for XGB regressor, as found in hp optimisation
    xgb = XGBRegressor(objective="reg:squarederror",
                       max_depth = study.best_params["max_depth"],
                       n_estimators = study.best_params["n_estimators"],
                       n_jobs = 40,
                       random_state = i)
    
    xgb.fit(np.array(train_PLEC), np.array(train_Potency))
    #Test the XGB model on the test molecules:
    predicted_potency = xgb.predict(np.array(test_PLEC))

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_xgb = pd.DataFrame({"Predicted_Potency": predicted_potency,
                               "Real_Potency": test_Potency,
                               "Real_Class": test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_xgb = result_xgb.sort_values(by='Predicted_Potency',ascending=False)
    result_xgb.to_csv("./screening-results/heldout_cluster_{}/PLEC/full_test_set/PfDHODH-rXGB-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [05:34<00:00, 33.50s/it]


In [12]:
#Train the XGB model on the training molecules, using optimal parameters:
for i in tqdm(range(10)):
    #Define the parameters for XGB regressor, as found in hp optimisation
    xgb = XGBRegressor(objective="reg:squarederror",
                       max_depth = study.best_params["max_depth"],
                       n_estimators = study.best_params["n_estimators"],
                       n_jobs = 40,
                       random_state = i)
    
    xgb.fit(np.array(train_PLEC), np.array(train_Potency))
    #Test the XGB model on the test molecules:
    predicted_potency = xgb.predict(np.array(test_actdecoys_test_PLEC))

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_xgb = pd.DataFrame({"Predicted_Potency": predicted_potency,
                               "Real_Potency": test_actdecoys_test_Potency,
                               "Real_Class": test_actdecoys_test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_xgb = result_xgb.sort_values(by='Predicted_Potency',ascending=False)
    result_xgb.to_csv("./screening-results/heldout_cluster_{}/PLEC/smaller_test_set/PfDHODH-rXGB-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [05:33<00:00, 33.30s/it]


#### 6.3 Support Vector Machine Regressor

In [13]:
def objective(trial: optuna.trial.Trial) -> float:
    # Define the search space using trial.suggest_* methods "
    C = trial.suggest_int("C", 1, 10)
    kernel = trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"])
    
    # Use these parameters to train and evaluate your model
    model = SVR(C = C,
                kernel = kernel)
    # Binary labels ONLY for stratification
    y_bin = (train_Potency >= 6).astype(int)

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    nef_scores = []
    # mse_scores = []

    for train_idx, val_idx in cv.split(train_PLEC, y_bin):
        X_tr, X_val = train_PLEC[train_idx], train_PLEC[val_idx]
        y_tr, y_val = train_Potency[train_idx], train_Potency[val_idx]

        # Train regression model on potency
        model.fit(X_tr, y_tr)

        # Predict potency
        y_pred = model.predict(X_val)

        # Compute NEF on validation fold
        nef = nef_score(y_val, y_pred, top_percent=10)
        nef_scores.append(nef)

        # mse = mean_squared_error(y_val, y_pred)
        # mse_scores.append(mse)
    
    return float(np.mean(nef_scores))

# Use HEBOSampler
module = optunahub.load_module("samplers/hebo")
sampler = module.HEBOSampler()

# Create and run the study
study = optuna.create_study(sampler=sampler, direction="maximize")
study.optimize(objective, n_trials=10)

print(study.best_trial.params, study.best_trial.value)

[I 2026-03-31 17:27:42,164] A new study created in memory with name: no-name-8d14bb5d-71bd-419c-9cc4-8eca3756c5b1
[I 2026-03-31 17:27:51,961] Trial 0 finished with value: 0.10606060606060606 and parameters: {'C': 6, 'kernel': 'sigmoid'}. Best is trial 0 with value: 0.10606060606060606.
[I 2026-03-31 17:28:00,535] Trial 1 finished with value: 0.9393939393939394 and parameters: {'C': 7, 'kernel': 'poly'}. Best is trial 1 with value: 0.9393939393939394.
[I 2026-03-31 17:28:10,146] Trial 2 finished with value: 0.07575757575757576 and parameters: {'C': 8, 'kernel': 'sigmoid'}. Best is trial 1 with value: 0.9393939393939394.
[I 2026-03-31 17:28:21,432] Trial 3 finished with value: 0.9545454545454546 and parameters: {'C': 10, 'kernel': 'rbf'}. Best is trial 3 with value: 0.9545454545454546.
[I 2026-03-31 17:28:33,445] Trial 4 finished with value: 0.9545454545454546 and parameters: {'C': 6, 'kernel': 'rbf'}. Best is trial 3 with value: 0.9545454545454546.
[I 2026-03-31 17:28:45,244] Trial 5 fi

{'C': 10, 'kernel': 'rbf'} 0.9545454545454546


In [14]:
for i in tqdm(range(10)):
    #Train the SVM model on the training molecules, using optimal parameters:
    svm = SVR(C = study.best_params['C'], kernel = study.best_params['kernel'])
    svm.fit(train_PLEC, train_Potency)

    #Test the SVM model on the test molecules:
    predicted_potency = svm.predict(test_PLEC)

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_svm  = pd.DataFrame({"Predicted_Potency": predicted_potency,
                                "Real_Potency": test_Potency,
                                "Real_Class": test_Class})
    result_svm = result_svm.sort_values(by='Predicted_Potency',ascending=False)
    result_svm.to_csv("./screening-results/heldout_cluster_{}/PLEC/full_test_set/PfDHODH-rSVM-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [03:03<00:00, 18.34s/it]


In [15]:
for i in tqdm(range(10)):
    #Train the SVM model on the training molecules, using optimal parameters:
    svm = SVR(C = study.best_params['C'], kernel = study.best_params['kernel'])
    svm.fit(train_PLEC, train_Potency)

    #Test the SVM model on the test molecules:
    predicted_potency = svm.predict(test_actdecoys_test_PLEC)

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_svm  = pd.DataFrame({"Predicted_Potency": predicted_potency,
                                "Real_Potency": test_actdecoys_test_Potency,
                                "Real_Class": test_actdecoys_test_Class})
    result_svm = result_svm.sort_values(by='Predicted_Potency',ascending=False)
    result_svm.to_csv("./screening-results/heldout_cluster_{}/PLEC/smaller_test_set/PfDHODH-rSVM-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [00:59<00:00,  5.97s/it]


# Train ML SFs on PLEC + Morgan fingerprints

In [16]:
# Define a function to generate Morgan fingerprints
def generate_morgan_fingerprints(smiles_list, radius=2, fpSize=512):
    morgan_fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, countSimulation=False, includeChirality=True, fpSize=fpSize)
    fingerprints = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        morgan_fp = morgan_fp_gen.GetFingerprint(mol)
        fp_vector = np.array(morgan_fp)
        fingerprints.append(fp_vector)
    return fingerprints

In [17]:
train_smiles = train_df['SMILES'].tolist()
full_test_smiles = full_test_df['SMILES'].tolist()
test_actdecoys_test_smiles = test_act_decoys_test_df['SMILES'].tolist()

# Generate training Morgan fingerprints
train_mfps = generate_morgan_fingerprints(train_smiles)
#Generate test Morgan fingerprints
test_mfps = generate_morgan_fingerprints(full_test_smiles)
test_actdecoys_test_mfps = generate_morgan_fingerprints(test_actdecoys_test_smiles)

In [18]:
train_features = [np.hstack((f1, f2)) for f1, f2 in zip(train_PLEC, train_mfps)]
test_features = [np.hstack((f1, f2)) for f1, f2 in zip(test_PLEC, test_mfps)]
test_actdecoys_test_features = [np.hstack((f1, f2)) for f1, f2 in zip(test_actdecoys_test_PLEC, test_actdecoys_test_mfps)]

## Random Forest Regressor

In [19]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 100,2000)
    criterion = trial.suggest_categorical("criterion", ["squared_error", "friedman_mse", "poisson"])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        criterion=criterion,
        max_features='sqrt',
        n_jobs=40, 
        random_state=42
    )
    # Binary labels ONLY for stratification
    y_bin = (train_Potency >= 6).astype(int)

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    nef_scores = []
    # mse_scores = []

    for train_idx, val_idx in cv.split(train_PLEC, y_bin):
        X_tr, X_val = train_PLEC[train_idx], train_PLEC[val_idx]
        y_tr, y_val = train_Potency[train_idx], train_Potency[val_idx]

        # Train regression model on potency
        model.fit(X_tr, y_tr)

        # Predict potency
        y_pred = model.predict(X_val)

        # Compute NEF on validation fold
        nef = nef_score(y_val, y_pred, top_percent=10)
        nef_scores.append(nef)

        # mse = mean_squared_error(y_val, y_pred)
        # mse_scores.append(mse)
    
    return float(np.mean(nef_scores))

    # return float(np.mean(nef_scores))

# Use HEBOSampler
module = optunahub.load_module("samplers/hebo")
sampler = module.HEBOSampler()

# Create and run the study
study = optuna.create_study(sampler=sampler, direction = 'maximize')
study.optimize(objective, n_trials=10)

print(study.best_trial.params, study.best_trial.value)

[I 2026-03-31 17:33:39,518] A new study created in memory with name: no-name-d375181e-9b27-4f3e-8827-a1466038b1c5
[I 2026-03-31 17:34:00,919] Trial 0 finished with value: 0.9696969696969697 and parameters: {'n_estimators': 1716, 'criterion': 'poisson'}. Best is trial 0 with value: 0.9696969696969697.
[I 2026-03-31 17:34:08,183] Trial 1 finished with value: 0.9545454545454546 and parameters: {'n_estimators': 569, 'criterion': 'friedman_mse'}. Best is trial 0 with value: 0.9696969696969697.
[I 2026-03-31 17:34:15,857] Trial 2 finished with value: 0.9696969696969697 and parameters: {'n_estimators': 598, 'criterion': 'poisson'}. Best is trial 0 with value: 0.9696969696969697.
[I 2026-03-31 17:34:29,684] Trial 3 finished with value: 0.9696969696969697 and parameters: {'n_estimators': 831, 'criterion': 'poisson'}. Best is trial 0 with value: 0.9696969696969697.
[I 2026-03-31 17:34:34,062] Trial 4 finished with value: 0.9545454545454546 and parameters: {'n_estimators': 109, 'criterion': 'pois

{'n_estimators': 1716, 'criterion': 'poisson'} 0.9696969696969697


In [20]:
# Evaluate on the full test set
for i in tqdm(range(10)):
    #Train the RF model on the training molecules, using optimal parameters:
    rf = RandomForestRegressor(n_estimators = study.best_params['n_estimators'], 
                               criterion = study.best_params['criterion'],
                               max_features = 'sqrt', n_jobs = 40, random_state=i)
    rf.fit(train_features, train_Potency)
    #Test the RF model on the test molecules:
    predicted_potency = rf.predict(test_features)
    
    #Get virtual screening results on the test molecules and export results to a csv file:
    result_rf = pd.DataFrame({"Predicted_Potency": predicted_potency,
                              "Real_Potency": test_Potency,
                              "Real_Class": test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_rf = result_rf.sort_values(by='Predicted_Potency',ascending=False)
    result_rf.to_csv("./screening-results/heldout_cluster_{}/PLEC+Morgan/full_test_set/PfDHODH-rRF-seed{}.csv".format(heldout_cluster,i+1))

100%|██████████| 10/10 [01:48<00:00, 10.82s/it]


In [21]:
# Evaluate on the test set containing test active decoys only (no training actives decoys)
for i in tqdm(range(10)):
    #Train the RF model on the training molecules, using optimal parameters:
    rf = RandomForestRegressor(n_estimators = study.best_params['n_estimators'], 
                               criterion = study.best_params['criterion'],
                               max_features = 'sqrt', n_jobs = 40, random_state=i)
    rf.fit(train_features, train_Potency)
    #Test the RF model on the test molecules:
    predicted_potency = rf.predict(test_actdecoys_test_features)

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_rf = pd.DataFrame({"Predicted_Potency": predicted_potency,
                              "Real_Potency": test_actdecoys_test_Potency,
                              "Real_Class": test_actdecoys_test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_rf = result_rf.sort_values(by='Predicted_Potency',ascending=False)
    result_rf.to_csv("./screening-results/heldout_cluster_{}/PLEC+Morgan/smaller_test_set/PfDHODH-rRF-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [01:39<00:00,  9.99s/it]


#### 6.2 Extreme Gradient Boosting Regressor

In [22]:
def objective(trial: optuna.trial.Trial) -> float:
    # Define the search space using trial.suggest_* methods "
    n_estimators = trial.suggest_int("n_estimators", 100,1000)
    max_depth = trial.suggest_categorical("max_depth", [5, 6, 7, 8, 9, 10])
    
    # Use these parameters to train and evaluate your model
    model = XGBRegressor(n_estimators = n_estimators,
                         max_depth = max_depth,
                         n_jobs = 40)
    
    # Binary labels ONLY for stratification
    y_bin = (train_Potency >= 6).astype(int)

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    nef_scores = []
    # mse_scores = []

    for train_idx, val_idx in cv.split(train_PLEC, y_bin):
        X_tr, X_val = train_PLEC[train_idx], train_PLEC[val_idx]
        y_tr, y_val = train_Potency[train_idx], train_Potency[val_idx]

        # Train regression model on potency
        model.fit(X_tr, y_tr)

        # Predict potency
        y_pred = model.predict(X_val)

        # Compute NEF on validation fold
        nef = nef_score(y_val, y_pred, top_percent=10)
        nef_scores.append(nef)

        # mse = mean_squared_error(y_val, y_pred)
        # mse_scores.append(mse)
    
    return float(np.mean(nef_scores))

    # return float(np.mean(nef_scores))

# Use HEBOSampler
module = optunahub.load_module("samplers/hebo")
sampler = module.HEBOSampler()

# Create and run the study
study = optuna.create_study(sampler=sampler, direction="maximize")
study.optimize(objective, n_trials=10)

print(study.best_trial.params, study.best_trial.value)

[I 2026-03-31 17:39:42,619] A new study created in memory with name: no-name-4b1872b1-3557-4131-a7e9-193818972781
[I 2026-03-31 17:41:00,404] Trial 0 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 957, 'max_depth': 7}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:41:49,139] Trial 1 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 237, 'max_depth': 7}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:42:39,969] Trial 2 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 327, 'max_depth': 9}. Best is trial 0 with value: 0.8939393939393939.


jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fitting GP
jitter = 1e-07
jitter = 1e-06
jitter = 9.999999999999999e-06
jitter = 9.999999999999999e-05
jitter = 0.001
jitter = 0.01
jitter = 0.1
jitter = 1.0
jitter = 10.0
jitter is too large, give up fittin

[I 2026-03-31 17:43:52,627] Trial 3 finished with value: 0.8636363636363636 and parameters: {'n_estimators': 626, 'max_depth': 6}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:44:46,579] Trial 4 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 329, 'max_depth': 9}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:45:36,646] Trial 5 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 231, 'max_depth': 7}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:46:46,032] Trial 6 finished with value: 0.8939393939393939 and parameters: {'n_estimators': 740, 'max_depth': 10}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:47:36,198] Trial 7 finished with value: 0.8787878787878788 and parameters: {'n_estimators': 340, 'max_depth': 5}. Best is trial 0 with value: 0.8939393939393939.
[I 2026-03-31 17:48:57,994] Trial 8 finished with value: 0.8939393939393939 and parameters: {'n_estimato

{'n_estimators': 957, 'max_depth': 7} 0.8939393939393939


In [23]:
#Train the XGB model on the training molecules, using optimal parameters:
for i in tqdm(range(10)):
    #Define the parameters for XGB regressor, as found in hp optimisation
    xgb = XGBRegressor(objective="reg:squarederror",
                       max_depth = study.best_params["max_depth"],
                       n_estimators = study.best_params["n_estimators"],
                       n_jobs = 40,
                       random_state = i)
    
    xgb.fit(np.array(train_features), np.array(train_Potency))
    #Test the XGB model on the test molecules:
    predicted_potency = xgb.predict(np.array(test_features))

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_xgb = pd.DataFrame({"Predicted_Potency": predicted_potency,
                               "Real_Potency": test_Potency,
                               "Real_Class": test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_xgb = result_xgb.sort_values(by='Predicted_Potency',ascending=False)
    result_xgb.to_csv("./screening-results/heldout_cluster_{}/PLEC+Morgan/full_test_set/PfDHODH-rXGB-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [06:51<00:00, 41.17s/it]


In [24]:
#Train the XGB model on the training molecules, using optimal parameters:
for i in tqdm(range(10)):
    #Define the parameters for XGB regressor, as found in hp optimisation
    xgb = XGBRegressor(objective="reg:squarederror",
                       max_depth = study.best_params["max_depth"],
                       n_estimators = study.best_params["n_estimators"],
                       n_jobs = 40,
                       random_state = i)

    xgb.fit(np.array(train_features), np.array(train_Potency))
    #Test the XGB model on the test molecules:
    predicted_potency = xgb.predict(np.array(test_actdecoys_test_features))

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_xgb = pd.DataFrame({"Predicted_Potency": predicted_potency,
                               "Real_Potency": test_actdecoys_test_Potency,
                               "Real_Class": test_actdecoys_test_Class})
    #Sort by the "Predicted_Potency" column in the descending order
    result_xgb = result_xgb.sort_values(by='Predicted_Potency',ascending=False)
    result_xgb.to_csv("./screening-results/heldout_cluster_{}/PLEC+Morgan/smaller_test_set/PfDHODH-rXGB-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [06:54<00:00, 41.42s/it]


#### 6.3 Support Vector Machine Regressor

In [25]:
def objective(trial: optuna.trial.Trial) -> float:
    # Define the search space using trial.suggest_* methods "
    C = trial.suggest_int("C", 1, 10)
    kernel = trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"])
    
    # Use these parameters to train and evaluate your model
    model = SVR(C = C,
                kernel = kernel)
    # Binary labels ONLY for stratification
    y_bin = (train_Potency >= 6).astype(int)

    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    nef_scores = []
    # mse_scores = []

    for train_idx, val_idx in cv.split(train_PLEC, y_bin):
        X_tr, X_val = train_PLEC[train_idx], train_PLEC[val_idx]
        y_tr, y_val = train_Potency[train_idx], train_Potency[val_idx]

        # Train regression model on potency
        model.fit(X_tr, y_tr)

        # Predict potency
        y_pred = model.predict(X_val)

        # Compute NEF on validation fold
        nef = nef_score(y_val, y_pred, top_percent=10)
        nef_scores.append(nef)

        # mse = mean_squared_error(y_val, y_pred)
        # mse_scores.append(mse)
    
    # return float(np.mean(mse_scores))

    return float(np.mean(nef_scores))

# Use HEBOSampler
module = optunahub.load_module("samplers/hebo")
sampler = module.HEBOSampler()

# Create and run the study
study = optuna.create_study(sampler=sampler, direction="maximize")
study.optimize(objective, n_trials=10)

print(study.best_trial.params, study.best_trial.value)

[I 2026-03-31 18:03:44,270] A new study created in memory with name: no-name-e77ad645-c5e7-4240-821c-9ea012a4a093
[I 2026-03-31 18:03:54,112] Trial 0 finished with value: 0.10606060606060606 and parameters: {'C': 6, 'kernel': 'sigmoid'}. Best is trial 0 with value: 0.10606060606060606.
[I 2026-03-31 18:04:03,618] Trial 1 finished with value: 0.27272727272727276 and parameters: {'C': 3, 'kernel': 'sigmoid'}. Best is trial 1 with value: 0.27272727272727276.
[I 2026-03-31 18:04:12,323] Trial 2 finished with value: 0.9393939393939394 and parameters: {'C': 4, 'kernel': 'poly'}. Best is trial 2 with value: 0.9393939393939394.
[I 2026-03-31 18:04:24,549] Trial 3 finished with value: 0.9090909090909092 and parameters: {'C': 1, 'kernel': 'rbf'}. Best is trial 2 with value: 0.9393939393939394.
[I 2026-03-31 18:04:36,675] Trial 4 finished with value: 0.9545454545454546 and parameters: {'C': 5, 'kernel': 'rbf'}. Best is trial 4 with value: 0.9545454545454546.
[I 2026-03-31 18:04:48,303] Trial 5 fi

{'C': 5, 'kernel': 'rbf'} 0.9545454545454546


In [26]:
for i in tqdm(range(10)):
    #Train the SVM model on the training molecules, using optimal parameters:
    svm = SVR(C = study.best_params['C'], kernel = study.best_params['kernel'])
    svm.fit(train_features, train_Potency)

    #Test the SVM model on the test molecules:
    predicted_potency = svm.predict(test_features)

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_svm  = pd.DataFrame({"Predicted_Potency": predicted_potency,
                                "Real_Potency": test_Potency,
                                "Real_Class": test_Class})
    result_svm = result_svm.sort_values(by='Predicted_Potency',ascending=False)
    result_svm.to_csv("./screening-results/heldout_cluster_{}/PLEC+Morgan/full_test_set/PfDHODH-rSVM-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [03:17<00:00, 19.76s/it]


In [27]:
for i in tqdm(range(10)):
    #Train the SVM model on the training molecules, using optimal parameters:
    svm = SVR(C = study.best_params['C'], kernel = study.best_params['kernel'])
    svm.fit(train_features, train_Potency)

    #Test the SVM model on the test molecules:
    predicted_potency = svm.predict(test_actdecoys_test_features)

    #Get virtual screening results on the test molecules and export results to a csv file:
    result_svm  = pd.DataFrame({"Predicted_Potency": predicted_potency,
                                "Real_Potency": test_actdecoys_test_Potency,
                                "Real_Class": test_actdecoys_test_Class})
    result_svm = result_svm.sort_values(by='Predicted_Potency',ascending=False)
    result_svm.to_csv("./screening-results/heldout_cluster_{}/PLEC+Morgan/smaller_test_set/PfDHODH-rSVM-seed{}.csv".format(heldout_cluster, i+1))

100%|██████████| 10/10 [01:02<00:00,  6.28s/it]


# Ensemble models

In [28]:
# for h_cluster in range(4):
#     dir = "./screening-results/heldout_cluster_{}/".format(h_cluster)
#     for seed in range(10):
#         RF_file = pd.read_csv(os.path.join(dir,"PLEC/smaller_test_set/PfDHODH-rRF-seed{}.csv"
#                                            .format(seed+1))).sort_values(by='Unnamed: 0',ascending=True, ignore_index=True)
#         XGB_file = pd.read_csv(os.path.join(dir,"PLEC/smaller_test_set/PfDHODH-rXGB-seed{}.csv"
#                                             .format(seed+1))).sort_values(by='Unnamed: 0',ascending=True, ignore_index=True)
#         SVM_file = pd.read_csv(os.path.join(dir,"PLEC/smaller_test_set/PfDHODH-rSVM-seed{}.csv"
#                                             .format(seed+1))).sort_values(by='Unnamed: 0',ascending=True, ignore_index=True)
#         new_df_data = {"RF_preds": RF_file["Predicted_Potency"],
#                   "XGB_preds": XGB_file["Predicted_Potency"],
#                   "SVM_preds": SVM_file["Predicted_Potency"],
#                   "Real_Potency": RF_file["Real_Potency"]}
#         new_df = pd.DataFrame(new_df_data)
#         new_df["Mean_Potency"] = new_df[["RF_preds","XGB_preds","SVM_preds"]].mean(axis=1)
#         new_df["Real_Class"] = RF_file["Real_Class"]
#         new_df = new_df.sort_values(by="Mean_Potency", ascending=False)
#         new_df.to_csv(os.path.join(dir,"ensemble/smaller_test_set/PfDHODH-ensemble-seed{}.csv".format(seed+1)))

## Evaluate Smina

In [29]:
# # Define a function to get train and test splits based on cluster IDs
# def get_train_test_split(df, heldout_cluster):
#     """
#     heldout_cluster: cluster ID to use as test actives (0, 1, 2, 3)
#     """
#     # --- ACTIVE SPLIT ---
#     train_actives = df[(df["activity"] == "Active") &
#                        (df["Cluster_ID"] != heldout_cluster)]

#     test_actives = df[(df["activity"] == "Active") &
#                       (df["Cluster_ID"] == heldout_cluster)]

#     # --- INACTIVE SPLIT ---
#     train_inactives = df[(df["activity"] == "Inactive") & 
#                          (df["Cluster_ID"] == -1)]

#     all_decoys = df[(df["activity"] == "Inactive") &
#                         (df["Cluster_ID"] != -1)]
    
#     test_act_decoys = df[(df["activity"]=="Inactive") & 
#                          (df["Cluster_ID"] == heldout_cluster)]

#     # --- FINAL MERGING ---
#     train_df = pd.concat([train_actives, train_inactives], ignore_index=True)
#     full_test_df  = pd.concat([test_actives, all_decoys], ignore_index=True)
#     test_act_decoys_test_df = pd.concat([test_actives, test_act_decoys], ignore_index=True)

#     return train_df, full_test_df, test_act_decoys_test_df

# # Define a function to load training and test features into separate np arrays
# def split_features(df, train_df, test_df):
#     " Provide a dataframe to split into training and test features based on train and test data molecule ids "
#     train_features = df.loc[train_df['mol_name']].values
#     test_features = df.loc[test_df['mol_name']].values
#     return {"train_features": train_features, "test_features": test_features}

In [30]:
# heldout_cluster = 3

In [31]:
# PfDHODH_data = pd.read_csv("../data/PfDHODH-data.csv")
# train_df, full_test_df, test_act_decoys_test_df = get_train_test_split(PfDHODH_data, heldout_cluster=heldout_cluster)

In [32]:
# # Extract Smina results to assess performance on the test set with actives from heldout cluster only
# Smina_results = pd.read_csv("../Smina/PfDHODH-Smina.csv")

# Smina_results[Smina_results['ID'].isin(full_test_df['mol_name'])].to_csv("../Smina/heldout_cluster_{}/full_test_set/PfDHODH-Smina.csv".format(heldout_cluster))
# Smina_results[Smina_results['ID'].isin(test_act_decoys_test_df['mol_name'])].to_csv("../Smina/heldout_cluster_{}/smaller_test_set/PfDHODH-Smina.csv".format(heldout_cluster))

## Evaluate CNN-Score

In [33]:
# # Extract CNN-Score results to assess performance on the test set with actives from heldout cluster only
# CNN_results = pd.read_csv("../CNN-Score/PfDHODH-CNN.csv")

# CNN_results[CNN_results['ID'].isin(full_test_df['mol_name'])].to_csv("../CNN-Score/heldout_cluster_{}/full_test_set/PfDHODH-CNN.csv".format(heldout_cluster))
# CNN_results[CNN_results['ID'].isin(test_act_decoys_test_df['mol_name'])].to_csv("../CNN-Score/heldout_cluster_{}/smaller_test_set/PfDHODH-CNN.csv".format(heldout_cluster))

# Train RF on PLEC + MF + physicochemical features

In [34]:
# train_features = [np.hstack((f1, f2, f3)) for f1, f2, f3 in zip(train_PLEC, train_mfps, train_physchem)]
# test_features = [np.hstack((f1, f2, f3)) for f1, f2, f3 in zip(test_PLEC, test_mfps, test_physchem)]

In [35]:
# def objective(trial: optuna.trial.Trial) -> float:
#     # Define the search space using trial.suggest_* methods
#     n_estimators = trial.suggest_int("n_estimators", 100,2000)
#     criterion = trial.suggest_categorical("criterion", ["squared_error", "friedman_mse", "poisson"])
    
#     # Here you would use these parameters to train and evaluate your model
#     model = RandomForestRegressor(n_estimators = n_estimators,
#                                   criterion=criterion, 
#                                   max_features = 'sqrt', 
#                                   n_jobs = 40)
#     model.fit(train_features, Train_Potency)
#     predicted_potency = model.predict(train_features)

#     loss = mean_squared_error(predicted_potency, Train_Potency)
#     return loss

# # Use HEBOSampler
# module = optunahub.load_module("samplers/hebo")
# sampler = module.HEBOSampler()

# # Create and run the study
# study = optuna.create_study(sampler=sampler, direction = 'maximize')
# study.optimize(objective, n_trials=10)

# print(study.best_trial.params, study.best_trial.value)

In [36]:
# for i in tqdm(range(10)):
#     #Train the RF model on the training molecules, using optimal parameters:
#     rf = RandomForestRegressor(n_estimators = study.best_params['n_estimators'], 
#                                #criterion = study.best_params['criterion'],
#                                max_features = 'sqrt', n_jobs = 40)
#     rf.fit(train_features, Train_Potency)
#     #Test the RF model on the test molecules:
#     predicted_potency = rf.predict(test_features)
    
#     #Get virtual screening results on the test molecules and export results to a csv file:
#     result_rf = pd.DataFrame({"Predicted_Potency": predicted_potency,
#                               "Real_Potency": Test_Potency,
#                               "Real_Class": Test_Class})
#     #Sort by the "Predicted_Potency" column in the descending order
#     result_rf = result_rf.sort_values(by='Predicted_Potency',ascending=False)
#     result_rf.to_csv("./screening-results/PLEC_Morgan_physchem/PfDHODH-opt_rRF-seed{}.csv".format(i+1))